# Dataset 7: missing-values and imputation experiment

This notebook studies how DCIts behaves when Dataset 7 contains missing values.
Two settings are tested:

1. Missing values are introduced in the training data.
2. Missing values are introduced in the test data.

For both settings, the notebook compares linear interpolation, forward/backward fill, and mean imputation across several missing-value ratios.


In [ ]:
import hashlib
import json
import os
import pickle
import random
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

# Load seminar-specific utilities and original DCIts source.
# This supports two layouts:
# 1. notebook inside DCIts/examples, where DCIts/src contains the utilities;
# 2. this inspection folder, where seminar-specific utilities are under
#    seminar_inspection/support_utils/src and the original DCIts clone is nearby.
def add_dcits_to_path():
    cwd = Path.cwd().resolve()
    search_bases = [cwd, *cwd.parents]

    utility_root = None
    dcits_root = None

    for base in search_bases:
        candidates = [
            base,
            base / "DCIts",
            base / "support_utils",
            base / "seminar_inspection" / "support_utils",
        ]
        for candidate in candidates:
            if utility_root is None and (candidate / "src" / "utils_impute.py").exists():
                utility_root = candidate
            if dcits_root is None and (candidate / "src" / "dcits.py").exists():
                dcits_root = candidate

    if utility_root is None:
        raise FileNotFoundError("Could not find src/utils_impute.py.")
    if dcits_root is None:
        raise FileNotFoundError("Could not find src/dcits.py from the original DCIts project.")

    for candidate in [str(utility_root), str(dcits_root)]:
        if candidate not in sys.path:
            sys.path.insert(0, candidate)

    return utility_root, dcits_root


UTILITY_ROOT, DCITS_ROOT = add_dcits_to_path()
print(f"Using seminar utilities from: {UTILITY_ROOT}")
print(f"Using DCIts source from: {DCITS_ROOT}")
from src.utils_impute import *

if torch.cuda.is_available():
    device = torch.device("cuda:0")
    print(f"Using {torch.cuda.get_device_name(device)}")
else:
    device = torch.device("cpu")
    print("CUDA is not available. Using CPU instead.")

ARTIFACT_ROOT = Path("artifacts")
RUN_NAME = "missing_values_imputation"
OUT_DIR = ARTIFACT_ROOT / RUN_NAME
DATA_DIR = OUT_DIR / "data"
FIG_DIR = OUT_DIR / "figures"

for path in (DATA_DIR, FIG_DIR):
    path.mkdir(parents=True, exist_ok=True)

METHOD_NAMES = {
    "linear": "Linear",
    "forward": "Forward fill",
    "mean": "Mean",
}


## Dataset 7 ground truth

The Dataset 7 generating equations are

$$
\begin{split}
X_{1,t} &= \alpha^\text{gt}_{1,1,1} X_{1,t-1} + \alpha^\text{gt}_{1,1,5} X_{1,t-5} + \epsilon_{1,t}, \\
X_{2,t} &= 1 + \alpha^\text{gt}_{2,1,2} X_{1,t-2} + \epsilon_{2,t}, \\
X_{3,t} &= \alpha^\text{gt}_{3,2,1} X_{2,t-1} + \alpha^\text{gt}_{3,4,4} X_{4,t-4} + \epsilon_{3,t}, \\
X_{4,t} &= 1 + \alpha^\text{gt}_{4,3,4} X_{3,t-4} + \alpha^\text{gt}_{4,5,1} X_{5,t-1} + \epsilon_{4,t}, \\
X_{5,t} &= \alpha^\text{gt}_{5,5,4} X_{5,t-4} + \alpha^\text{gt}_{5,2,1} X_{2,t-1} + \epsilon_{5,t}.
\end{split}
$$


In [ ]:
# Reproducibility
seed = 1000
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)

# Missing-value experiment grid
missing_ratios = [0.01, 0.02, 0.05, 0.10]
imputation_methods = ["linear", "forward", "mean"]

# Data-generation parameters
mean = 0.0
std = 0.1
noise_frequency = 1.0
ts_length = 20000
burn_in = 1000

# DCIts parameters
n_runs = 5
window_length_gp = 5
temperature = 1.0
order = [1, 1]

# Dataset dimensions
no_of_timeseries_gp = 5
dataset_name = "Dataset 7"
series_names = [f"X{i}" for i in range(1, no_of_timeseries_gp + 1)]

# Ground-truth tensor: [target, source, lag_index], where lag_index 0 means lag 1.
ground_truth_alpha = torch.zeros(no_of_timeseries_gp, no_of_timeseries_gp, window_length_gp)
ground_truth_alpha[0, 0, 0] = 1 / 4
ground_truth_alpha[0, 0, 4] = 3 / 4
ground_truth_alpha[1, 0, 1] = -1
ground_truth_alpha[2, 1, 0] = 1
ground_truth_alpha[2, 3, 3] = 1
ground_truth_alpha[3, 2, 3] = -2 / 7
ground_truth_alpha[3, 4, 0] = -5 / 7
ground_truth_alpha[4, 4, 3] = 12 / 22
ground_truth_alpha[4, 1, 0] = 10 / 22

alpha_mask = (ground_truth_alpha != 0).float()
ground_truth_bias = torch.tensor([0, 1, 0, 1, 0])

gt_alphas = [
    (0, 0, 0, 1 / 4),
    (0, 0, 4, 3 / 4),
    (1, 0, 1, -1),
    (2, 1, 0, 1),
    (2, 3, 3, 1),
    (3, 2, 3, -2 / 7),
    (3, 4, 0, -5 / 7),
    (4, 4, 3, 12 / 22),
    (4, 1, 0, 10 / 22),
]

time_series = dataset(
    ts_length,
    ground_truth_alpha,
    ground_truth_bias,
    noise_frequency=noise_frequency,
    mu=mean,
    sigma=std,
    seed=seed,
)
print("time_series shape:", tuple(time_series.shape))


In [ ]:
def to_numpy(x):
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.asarray(x)


def tensor_hash(tensor):
    arr = np.ascontiguousarray(to_numpy(tensor))
    return hashlib.sha256(arr.tobytes()).hexdigest()


def make_train_config():
    return {
        "verbose": False,
        "device": device,
        "learning_rate": 1e-3,
        "scheduler_patience": 5,
        "early_stopping_modifier": 2,
        "criterion": nn.MSELoss(),
    }


def cache_path_for(scenario, ratio, method, train_config):
    safe_config = dict(train_config)
    safe_config["device"] = str(safe_config["device"])
    safe_config["criterion"] = safe_config["criterion"].__class__.__name__

    payload = {
        "scenario": scenario,
        "time_series_shape": list(time_series.shape),
        "time_series_hash": tensor_hash(time_series),
        "n_runs": n_runs,
        "window_length": window_length_gp,
        "temperature": temperature,
        "order": order,
        "train_config": safe_config,
        "missing_ratio": ratio,
        "impute_method": method,
        "seed": seed,
    }
    key = hashlib.sha256(json.dumps(payload, sort_keys=True).encode("utf-8")).hexdigest()[:16]
    ratio_name = str(ratio).replace(".", "p")
    return DATA_DIR / f"{scenario}_{method}_ratio_{ratio_name}_{key}.pkl"


def alpha_corr_with_ground_truth(stats):
    # DCIts stores the model lag axis reversed relative to the ground-truth tensor.
    estimated = np.flip(to_numpy(stats["alpha"][1]["mean"]), axis=2)
    truth = to_numpy(ground_truth_alpha)
    return np.corrcoef(estimated.flatten(), truth.flatten())[0, 1]


def run_imputation_experiment(scenario, train_impute, test_impute):
    train_config = make_train_config()
    rows = []
    stats_by_combo = {}

    for ratio in missing_ratios:
        for method in imputation_methods:
            print(f"{scenario}: ratio={ratio}, method={method}")
            path = cache_path_for(scenario, ratio, method, train_config)

            if path.exists():
                with open(path, "rb") as f:
                    cache = pickle.load(f)
                results = cache["results"]
                stats = cache["stats"]
                print(f"  loaded cache: {path.name}")
            else:
                results = collect_multiple_runs(
                    n_runs=n_runs,
                    time_series=time_series,
                    window_size=window_length_gp,
                    temperature=temperature,
                    order=order,
                    config=train_config,
                    seed=seed,
                    verbose=False,
                    train_impute=train_impute,
                    test_impute=test_impute,
                    missing_ratio=ratio,
                    impute_method=method,
                )
                stats = calculate_multiple_run_statistics(results)
                with open(path, "wb") as f:
                    pickle.dump({"results": results, "stats": stats}, f, protocol=pickle.HIGHEST_PROTOCOL)
                print(f"  saved cache: {path.name}")

            rmse = float(np.sqrt(results["summary"]["mean_test_loss"]))
            alpha_corr = float(alpha_corr_with_ground_truth(stats))
            stats_by_combo[(method, ratio)] = stats
            rows.append({
                "scenario": scenario,
                "missing_ratio": ratio,
                "method": method,
                "rmse": rmse,
                "alpha_corr": alpha_corr,
            })
            print(f"  RMSE={rmse:.6f}, alpha_corr={alpha_corr:.3f}")

    summary = pd.DataFrame(rows)
    summary.to_csv(DATA_DIR / f"{scenario}_summary.csv", index=False)
    return {"summary": summary, "stats": stats_by_combo}


In [ ]:
def plot_metric_by_method(summary, metric, title, ylabel, filename):
    fig, ax = plt.subplots(figsize=(8, 6))
    for method in imputation_methods:
        method_rows = summary[summary["method"] == method]
        ax.plot(
            method_rows["missing_ratio"],
            method_rows[metric],
            marker="o",
            linewidth=2,
            label=METHOD_NAMES.get(method, method),
        )
    ax.set_xlabel("missing-value ratio")
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.3)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / filename, bbox_inches="tight", dpi=300)
    plt.show()
    plt.close(fig)


def flip_lag(lag):
    return window_length_gp - lag - 1


def plot_alpha_by_source(stats_by_combo, scenario):
    grouped = {}
    for target, source, lag, gt_val in gt_alphas:
        grouped.setdefault(source, []).append((target, source, lag, gt_val))

    for source in sorted(grouped):
        fig, axes = plt.subplots(1, len(imputation_methods), figsize=(18, 5), sharey=True)
        fig.suptitle(f"{scenario}: alpha estimates from {series_names[source]}")

        for ax, method in zip(axes, imputation_methods):
            for target, source, lag, gt_val in grouped[source]:
                lag_model = flip_lag(lag)
                mean_vals = []
                std_vals = []
                for ratio in missing_ratios:
                    stats = stats_by_combo[(method, ratio)]
                    mean_vals.append(stats["alpha"][1]["mean"][target, source, lag_model])
                    std_vals.append(stats["alpha"][1]["std"][target, source, lag_model])

                label = rf"$\alpha_{{{target+1},{source+1},{lag+1}}}$"
                ax.errorbar(missing_ratios, mean_vals, yerr=std_vals, marker="o", capsize=3, label=label)
                ax.axhline(gt_val, color="black", linestyle="--", linewidth=1)

            ax.set_title(METHOD_NAMES.get(method, method))
            ax.set_xlabel("missing-value ratio")
            ax.grid(True, alpha=0.3)

        axes[0].set_ylabel(r"$\alpha$ mean +/- std")
        handles, labels = axes[0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="upper right")
        fig.tight_layout(rect=[0, 0, 0.92, 0.95])
        fig.savefig(FIG_DIR / f"{scenario}_alpha_from_{series_names[source]}.png", bbox_inches="tight", dpi=300)
        plt.show()
        plt.close(fig)


def plot_alpha_rmse_by_source(stats_by_combo, scenario):
    grouped = {}
    for target, source, lag, gt_val in gt_alphas:
        grouped.setdefault(source, []).append((target, source, lag, gt_val))

    for source in sorted(grouped):
        fig, ax = plt.subplots(figsize=(10, 6))

        for target, source, lag, gt_val in grouped[source]:
            lag_model = flip_lag(lag)
            for method in imputation_methods:
                rmse_vals = []
                for ratio in missing_ratios:
                    stats = stats_by_combo[(method, ratio)]
                    est = stats["alpha"][1]["mean"][target, source, lag_model]
                    rmse_vals.append(abs(est - gt_val))

                ax.plot(
                    missing_ratios,
                    rmse_vals,
                    marker="o",
                    label=rf"{METHOD_NAMES.get(method, method)} $\alpha_{{{target+1},{source+1},{lag+1}}}$",
                )

        ax.set_xlabel("missing-value ratio")
        ax.set_ylabel(r"absolute alpha error")
        ax.set_title(f"{scenario}: alpha error from {series_names[source]}")
        ax.grid(True, alpha=0.3)
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIG_DIR / f"{scenario}_alpha_error_from_{series_names[source]}.png", bbox_inches="tight", dpi=300)
        plt.show()
        plt.close(fig)


def plot_all_summaries(experiment, scenario):
    summary = experiment["summary"]
    stats_by_combo = experiment["stats"]

    plot_metric_by_method(
        summary,
        metric="rmse",
        title=f"{scenario}: prediction error by imputation method",
        ylabel="RMSE",
        filename=f"{scenario}_rmse_by_method.png",
    )
    plot_metric_by_method(
        summary,
        metric="alpha_corr",
        title=f"{scenario}: alpha correlation by imputation method",
        ylabel="alpha correlation",
        filename=f"{scenario}_alpha_corr_by_method.png",
    )
    plot_alpha_by_source(stats_by_combo, scenario)
    plot_alpha_rmse_by_source(stats_by_combo, scenario)


## Missing values in the training set

In this setting, missing values are introduced before training. The learned model is then evaluated on clean test data.


In [ ]:
train_missing = run_imputation_experiment(
    scenario="missing_in_train",
    train_impute=True,
    test_impute=False,
)
train_missing["summary"]


In [ ]:
plot_all_summaries(train_missing, "missing_in_train")


## Missing values in the test set

In this setting, the model is trained on clean data and missing values are introduced during testing.


In [ ]:
test_missing = run_imputation_experiment(
    scenario="missing_in_test",
    train_impute=False,
    test_impute=True,
)
test_missing["summary"]


In [ ]:
plot_all_summaries(test_missing, "missing_in_test")


## Combined result table


In [ ]:
combined_summary = pd.concat(
    [train_missing["summary"], test_missing["summary"]],
    ignore_index=True,
)
combined_summary.to_csv(DATA_DIR / "combined_imputation_summary.csv", index=False)
combined_summary
